# Exploring the Tabula Muris

The Tabula Muris is a publicly-available compendium of single cell sequencing data from from mice. 

The data were obtained by sequencing nearly 100,000 cells from 20 organs and tissues from mice (Mus Musculus)

In the original paper describing The Tabula Muris, the authors used two different sequencing technologies to characterise gene expression in many different tissues. The authors measuredAlthough the data was generated from many different cell types contained within their samples, the authors were able to use advanced analytical techniques to group the different cells into clusters of cells with shared gene expression profiles. Using known gene expression profiles (marker genes), the authors were able to conclude identify many of the different cell types present within their samples. 

Luckily for us, one of the cell types that the Tabula Muris consortium identified was endothelial cells. You are going to extract the endothelial cell transcriptomes from this dataset and use this data to create a map of the endothelial cell ion channel or GPCR toolkit!

## References and resources

*Full description of the Tabula Muris Project, interactive data and references.*

**Tabula Muris Project at The Chan Zuckerberg Biohub**

> **Link:** https://www.czbiohub.org/tabula-muris/ & https://tabula-muris.ds.czbiohub.org/

> **Description:** These websites provides some interactive tools to visualise the Tabula Muris data. You can also find links to all of the original data and example analysis files.

**Chan Zuckerberg Tabula Muris single-cell sequencing workshop**

> **Link:** https://chanzuckerberg.github.io/scRNA-python-workshop/intro/about.html

> **Description:** These websites provides some interactive tools to visualise the Tabula Muris data. You can also find links to all of the original data and example analysis files.

**Alexander Chervov's Kaggle Notebooks**

> **Link:** https://www.kaggle.com/alexandervc

> **Description:** These Kaggle notebooks show how to use Python to explore the Tabula Muris dataset.

**Original Reference**

Tabula Muris: https://www.nature.com/articles/s41586-018-0590-4 (03 October 2018).

**Additional References**

Tabula Senis 1: https://www.nature.com/articles/s41586-020-2496-1

Tabula Senis 2: https://elifesciences.org/articles/62293

Tabula Sapiens: https://www.science.org/doi/10.1126/science.abl4896

Another useful EC paper: https://www.ahajournals.org/doi/10.1161/CIRCULATIONAHA.119.041433

## The present notebook
The present notebook loads in the smart-seq data set and saves it in a useful format. It combines the raw data counts with the annotations file. 

Really, it does two things:

* Loads in the raw scRNAseq counts csv files.
* Loads in the annotation data (the files that contain information to identify cells)
* Merges the above into a "AnnData" object - a special file format that can be manipulated using various functions in Python







**DO NOT CHANGE ANYTHING IN THIS NOTEBOOK. JUST RUN IT TO SAVE THE DATA INTO A DIFFERENT FORMAT THAT WE WILL USE LATER**

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Import additional Python modules that we need

In [ ]:
!pip install scanpy
import scanpy
import anndata

import scipy 


import time
t0start = time.time()

import pandas
import numpy
import os
import sys

import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 70
# plt.style.use('dark_background')

import seaborn as sns

from sklearn.decomposition import PCA

### Load the annotation data

#### This is the same file we looked at in the first Kaggle Workbook. We need it to match up the single cell sequencing data to cell type/tissue/mouse identifiers.

In [ ]:
# Define the overall FACs folder:

path_data2 = '/kaggle/input/scrnaseq-tabula-muris-mouse-85-000-cells/Single-cell RNA-seq data from Smart-seq2 sequencing of FACS sorted cells (v2)/'
file_list2 = os.listdir(path_data2)
print()
print('Annotation data:')
file_list3 = list( filter(lambda x: ('.csv' in x ) and ('FACS/' not in x), file_list2 ) )
print(file_list3)

In [ ]:
# Define the annotation file
annotation_file = path_data2 + file_list3[1]


# Define the datatypes

datatypes = {'Neurog3>0_raw': "string", 'Neurog3>0_scaled': "string", 'cell': "string", 'cell_ontology_class': "string",
       'cell_ontology_id': "string", 'cluster.ids': int, 'free_annotation': "string", 'mouse.id': "string",
       'mouse.sex': "string", 'plate.barcode': "string", 'subsetA': int, 'subsetA_cluster.ids': "string",
       'subsetB': int, 'subsetB_cluster.ids': "string", 'subsetC': int, 'subsetC_cluster.ids': "string",
       'subsetD': int, 'subsetD_cluster.ids': "string", 'subsetE': int, 'subsetE_cluster.ids': "string",
       'subtissue': "string", 'tissue': "string", 'tissue_tSNE_1': float, 'tissue_tSNE_2': float}


# Read in the csv file as a Pandas dataframe

df = pandas.read_csv(annotation_file, dtype=datatypes)#,index_col = 0)
df_annotations = df.copy()
print(list(df.columns))
df


# Lets have a little look at the first 5 rows of the dataframe

print('File: "annotations_facs.csv". ', 'Dataframe shape: ', df.shape )
display(df.head(5))
print()

### Load the smart-seq2 data

#### This is the files that contain all of the single cell sequencing data (the gene expression values) for all of the cells analysed in the Tabula Muris paper.

In [ ]:
# Define the FACs dataset folder:
path_name = '/kaggle/input/scrnaseq-tabula-muris-mouse-85-000-cells/Single-cell RNA-seq data from Smart-seq2 sequencing of FACS sorted cells (v2)/FACS/'

file_list = os.listdir(path_name)
print(file_list)

# Read all of the data files in a loop

In [ ]:
t0 = time.time()

verbose = 10

for i,fn in enumerate(file_list): #fn = filename
    str_data_inf = 'Tabula Muris '   + fn.split('-')[0]
    str_data_inf = 'smart-seq2 ' + fn.split('-')[0]
    df = pandas.read_csv(path_name + fn , index_col = 0)
    df=df.T
    a = scanpy.AnnData(df)
    if verbose >= 1:
        print(i, fn, df.shape, a.shape)
        print(a)
        
    scanpy.pp.filter_cells(a, min_genes=200)
    a.X = scipy.sparse.csr.csr_matrix(a.X) # Convert to sparse matrix 
    
    if i == 0:
        adata = a.copy()
    else:
        adata= adata.concatenate(a,batch_key=None, index_unique = None)
    print(i, len(set(adata.obs.index)), adata.shape )
    print()
        
    if i>=100:
        break

print('%.1f'%(-t0+time.time()), ' seconds passed' )
        
adata 

In [ ]:
# Checks that no in appropriate duplicates: 
print( len(set(adata.obs.index)), adata.shape,  df_annotations['cell'].nunique(), df_annotations.shape )
print( len( set(adata.obs.index) & set( df_annotations['cell'] ) ), adata.shape )

In [ ]:
# First we delete those cells from adata which are not present in the annotations file
m = adata.obs.index.isin(df_annotations['cell'] )
if verbose >= 100:
    print(m.sum())
adata = adata[m]
if verbose >= 100:
    print( adata )
    display( adata.obs.head(3) )

In [ ]:
# Now merge the annotations data into the Adata object: 
obs_new = pandas.merge(adata.obs,df_annotations, how = 'inner', left_index=True, right_on = 'cell' )
obs_new.set_index('cell',inplace = True )
obs_new

# let us put column "tissue" on the first place, in a bit tricky way: 
adata.obs = adata.obs.join(obs_new['tissue'] )
adata.obs['Cell type'] = obs_new['cell_ontology_class'] # 'Cell type' is more common name for us
adata.obs = adata.obs.join(obs_new.drop(columns = ['n_genes', 'tissue']))
adata

# Let's take a brief look at the data

In [ ]:
print('Look at count matrix. We see integers - that confirms - data are raw-counts, not preprocessed expressions')
adata.X.sum(), type(adata.X), numpy.asarray(adata.X.sum(axis = 1)).ravel()[:10]

### Create a plot showing the top 20 genes by expression levels.

To make this graph we will use scanpy. Scanpy is a Python package for analysing single cell sequencing data. Read more about what it does and what graphs it can create here:

* https://scanpy.readthedocs.io/en/stable/tutorials.html

The "save="XXXX" ensures that the plot is saved to the output folder (on the right hand side). Make sure you can download the plot!

In [ ]:

scanpy.pl.highest_expr_genes(adata, n_top=20, save="_top_20_unfiltered")



With scanpy, we used a single line of Python code to create this plot...

**But we can also easily change the plot layout with scanpy.set_figure_params**

* https://scanpy.readthedocs.io/en/stable/generated/scanpy.set_figure_params.html

In [ ]:
scanpy.set_figure_params(dpi=150, dpi_save=300, frameon=True, fontsize=10, figsize=(4,4), format='pdf')
# This sets the parameters for all future figures.


scanpy.pl.highest_expr_genes(adata, n_top=20, save="_top_20_unfiltered")


### We can also use scanpy to calculate quality control metrics

Read more about this here:

* https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.calculate_qc_metrics.html


In [ ]:
# Calculate some quality control metrics (works out how many cells express how many genes, etc)

scanpy.pp.calculate_qc_metrics(adata,  percent_top=None, log1p=False, inplace=True)
    # Calculates statistics on both cells and genes:
    # for cells: 'n_genes_by_counts', 'total_counts'
    # for genes: 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    # If parameter: inplace = True, then these statistics will be added to adata

print(adata)

# You can see below, that a number of new 


In [ ]:
# Lets plot the quality control statistics

scanpy.pl.violin(adata, ['n_genes_by_counts', 'total_counts'], jitter=0.4, multi_panel=True, save="_qc_metrics")
#scanpy.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')


display(adata.obs.describe() ) # cellwise statistics   

display(adata.var.describe()) # genewise statistics

# Save adata file for further use

In [ ]:
adata.uns['GSE'] =  'GSE109774'
adata.uns['Brief Info'] = 'Tabula Muris V2 smart-seq2 - Endothelial cell data'
adata.uns['Link'] = r'https://www.czbiohub.org/tabula-muris/'
adata.uns['Data Link'] = r'https://figshare.com/projects/Tabula_Muris_Transcriptomic_characterization_of_20_organs_and_tissues_from_Mus_musculus_at_single_cell_resolution/27733'


adata.uns['Date'] =  '2022 09 06'

adata.uns['Cancer Cells'] =  0

adata.uns['Organism'] = 'Mouse'
adata.uns['Cell type'] = 'All'
adata.uns['Cell further info'] = '20Tissues8Mices'
adata.uns['Year'] = 2018
adata.uns['Cell Count'] = adata.X.shape[0]
adata.uns['Source'] = 'CZI'
adata.uns['Counts or preprocessing'] = 'Counts'
adata.uns['First author'] = 'Barres'
adata.uns['Last author'] = 'WyssCoray'

adata.uns['Paper Link'] = r'https://www.nature.com/articles/s41586-018-0590-4'
adata.uns['Paper Title'] = 'Single-cell transcriptomics of 20 mouse organs creates a Tabula Muris'
adata.uns['Abstract'] = '''Here we present a compendium of single-cell transcriptomic data from the model organism Mus musculus that comprises more than 100,000 cells from 20 organs and tissues. These data represent a new resource for cell biology, reveal gene expression in poorly characterized cell populations and enable the direct and controlled comparison of gene expression in cell types that are shared between tissues, such as T lymphocytes and endothelial cells from different anatomical locations. Two distinct technical approaches were used for most organs: one approach, microfluidic droplet-based 3′-end counting, enabled the survey of thousands of cells at relatively low coverage, whereas the other, full-length transcript analysis based on fluorescence-activated cell sorting, enabled the characterization of cell types with high sensitivity and coverage. The cumulative data provide the foundation for an atlas of transcriptomic cell biology.'''
adata.uns['PMID'] = 30283141
    
adata.uns['Comment'] = r'Reformating scripts to h5ad can be found at: https://www.kaggle.com/alexandervc/4-tabula-muris-merge-all-smart-seq2-data' 

adata.uns['Technology'] =  'smart-seq2'

In [ ]:
# without that explicit transformation scanpy cannot save anndata, crashing by error
# some columns contains both NaN , logical True False - that probably causes a problem
for c in ['Neurog3>0_raw','Neurog3>0_scaled','subsetE', 'cell_ontology_id', 'channel', 'cluster.ids', 'free_annotation', 'mouse.id', 'mouse.sex', 'subsetA', 'subsetA_cluster.ids', 'subsetB', 'subsetB_cluster.ids', 'subsetC', 'subsetC_cluster.ids', 'subsetD', 'subsetD_cluster.ids', 'subtissue',]:
    if c not in adata.obs.columns: continue
    adata.obs[c] =    [str(t) for t in adata.obs[c] ]
    
fn = 'TabMurisSmartSeq2V2_Mouse_AllCells.h5ad' 
import time
t0 = time.time()
adata.write_h5ad(fn,compression='gzip')
print('%.1f'%(-t0+time.time()), ' seconds passed' )
adata  

In [ ]:
print(adata.obs['tissue'].unique())